In [1]:
import sys
sys.path.append('..')
from parser.split import build_train_test_split

ratings, train, held_out = build_train_test_split()
print(f"Train: {len(train)}, Held-out: {len(held_out)}")

Train: 80896, Held-out: 19940


In [2]:
import pandas as pd

movies = pd.read_csv("../sample_data/inputs/ml-latest-small/movies.csv")
movies['genres'].head(10)

0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: object

In [3]:
movies['genres_cleaned'] = movies['genres'].str.replace('|', ' ', regex=False) ## replacing | with spaces
movies[['title', 'genres', 'genres_cleaned']].head(10)

,title,genres,genres_cleaned
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy
5,Heat (1995),Action|Crime|Thriller,Action Crime Thriller
6,Sabrina (1995),Comedy|Romance,Comedy Romance
7,Tom and Huck (1995),Adventure|Children,Adventure Children
8,Sudden Death (1995),Action,Action
9,GoldenEye (1995),Action|Adventure|Thriller,Action Adventure Thriller


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer ## TF-IDF vectorization 

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies['genres_cleaned'])

print(genre_matrix.shape)

(9742, 24)


In [5]:
## Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

content_similarity = cosine_similarity(genre_matrix)
print(content_similarity.shape)

(9742, 9742)


In [6]:
## Sanity check on a single row
## Finding movies similar to toy story movie


movie_idx = movies[movies['title'].str.contains('Toy Story', case=False)].index[0]
similar_scores = content_similarity[movie_idx]

similar_indices = similar_scores.argsort()[::-1][1:11]
movies.iloc[similar_indices][['title', 'genres']]

,title,genres
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
8927,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
9430,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy


In [7]:
# Content-based score for one user

user_id = 1
liked_movies = train[(train['userId'] == user_id) & (train['rating'] >= 4)]['movieId'].tolist()
liked_indices = movies[movies['movieId'].isin(liked_movies)].index.tolist()

content_scores = content_similarity[liked_indices].mean(axis=0)
print(content_scores.shape)

(9742,)


In [8]:
from scoring.collaborative_filtering import build_cf_model

model, user_id_map, movie_id_map, movie_idx_to_id, user_item_matrix = build_cf_model(train)

c:\Users\Himanshu Jain\Downloads\recommendation-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Himanshu Jain\Downloads\recommendation-system\.venv\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:00<00:00, 48.26it/s]


In [9]:
content_scores_series = pd.Series(content_scores, index=movies['movieId'])

user_idx = user_id_map[user_id]
cf_scores_full = model.user_factors[user_idx] @ model.item_factors.T
cf_scores_series = pd.Series(cf_scores_full, index=[movie_idx_to_id[i] for i in range(len(movie_idx_to_id))])

print(content_scores_series.shape, cf_scores_series.shape)

(9742,) (8246,)


In [10]:
## Combining the CF score and the content based score
combined = pd.DataFrame({'content': content_scores_series, 'cf': cf_scores_series}).dropna()

combined['content_norm'] = (combined['content'] - combined['content'].min()) / (combined['content'].max() - combined['content'].min())
combined['cf_norm'] = (combined['cf'] - combined['cf'].min()) / (combined['cf'].max() - combined['cf'].min())

combined['hybrid_score'] = 0.5 * combined['content_norm'] + 0.5 * combined['cf_norm']
combined.sort_values('hybrid_score', ascending=False).head(10)

,content,cf,content_norm,cf_norm,hybrid_score
2005,0.373886,0.961917,0.945482,0.855761,0.900621
1291,0.315183,1.198416,0.797036,0.976590,0.886813
2617,0.337079,1.076064,0.852406,0.914079,0.883243
733,0.322087,1.075344,0.814493,0.913711,0.864102
2947,0.322087,1.051540,0.814493,0.901550,0.858021
367,0.313018,1.035862,0.791560,0.893540,0.842550
1275,0.322536,0.984610,0.815628,0.867355,0.841491
1197,0.355072,0.815743,0.897907,0.781079,0.839493
2406,0.340522,0.873888,0.861113,0.810785,0.835949
2115,0.322536,0.947404,0.815628,0.848346,0.831987


In [11]:
## createing a reusable function
def get_hybrid_recommendations(user_id, k=10, weight=0.5):
    liked_movies = train[(train['userId'] == user_id) & (train['rating'] >= 4)]['movieId'].tolist()
    already_rated = train[train['userId'] == user_id]['movieId'].tolist()
    
    if len(liked_movies) == 0 or user_id not in user_id_map:
        return []
    
    liked_indices = movies[movies['movieId'].isin(liked_movies)].index.tolist()
    content_scores = content_similarity[liked_indices].mean(axis=0)
    content_scores_series = pd.Series(content_scores, index=movies['movieId'])
    
    user_idx = user_id_map[user_id]
    cf_scores_full = model.user_factors[user_idx] @ model.item_factors.T
    cf_scores_series = pd.Series(cf_scores_full, index=[movie_idx_to_id[i] for i in range(len(movie_idx_to_id))])
    
    combined = pd.DataFrame({'content': content_scores_series, 'cf': cf_scores_series}).dropna()
    combined['content_norm'] = (combined['content'] - combined['content'].min()) / (combined['content'].max() - combined['content'].min())
    combined['cf_norm'] = (combined['cf'] - combined['cf'].min()) / (combined['cf'].max() - combined['cf'].min())
    combined['hybrid_score'] = weight * combined['content_norm'] + (1 - weight) * combined['cf_norm']
    
    combined = combined[~combined.index.isin(already_rated)]
    
    return combined.sort_values('hybrid_score', ascending=False).head(k).index.tolist()

get_hybrid_recommendations(1, k=10)

[2294, 380, 588, 1036, 153, 780, 3114, 3438, 10, 1610]

In [12]:
from scoring.evaluation import build_held_out_liked, get_hits, get_precision_recall

all_users = ratings['userId'].unique()
held_out_liked = build_held_out_liked(held_out)

In [13]:
cf_recommendations_by_user = {}
for user_id in all_users:
    if user_id not in user_id_map:
        continue
    user_idx = user_id_map[user_id]
    recommended = model.recommend(user_idx, user_item_matrix[user_idx], N=10)
    cf_recommendations_by_user[user_id] = [movie_idx_to_id[idx] for idx in recommended[0]]

cf_hit_rate = get_hits(cf_recommendations_by_user, held_out_liked, k=10)
cf_precision, cf_recall = get_precision_recall(cf_recommendations_by_user, held_out_liked, k=10)

In [14]:
hybrid_recommendations_by_user = {user: get_hybrid_recommendations(user, k=10) for user in all_users}

hybrid_hit_rate = get_hits(hybrid_recommendations_by_user, held_out_liked, k=10)
hybrid_precision, hybrid_recall = get_precision_recall(hybrid_recommendations_by_user, held_out_liked, k=10)

print(f"Hybrid                  — Hit-rate@10: {hybrid_hit_rate:.2%}, Precision@10: {hybrid_precision:.2%}, Recall@10: {hybrid_recall:.2%}")
print(f"Collaborative Filtering — Hit-rate@10: {cf_hit_rate:.2%}, Precision@10: {cf_precision:.2%}, Recall@10: {cf_recall:.2%}")

Hybrid                  — Hit-rate@10: 39.59%, Precision@10: 6.01%, Recall@10: 7.97%
Collaborative Filtering — Hit-rate@10: 42.13%, Precision@10: 6.82%, Recall@10: 7.97%


In [15]:
## tuning the weights associated with the CF filtering and the Content based recommendation system
for w in [0.1, 0.2, 0.3, 0.4, 0.5]:
    hybrid_recs = {user: get_hybrid_recommendations(user, k=10, weight=w) for user in all_users}
    hr = get_hits(hybrid_recs, held_out_liked, k=10)
    prec, rec = get_precision_recall(hybrid_recs, held_out_liked, k=10)
    print(f"weight={w} — Hit-rate@10: {hr:.2%}, Precision@10: {prec:.2%}, Recall@10: {rec:.2%}")

weight=0.1 — Hit-rate@10: 43.65%, Precision@10: 7.01%, Recall@10: 8.56%
weight=0.2 — Hit-rate@10: 43.15%, Precision@10: 6.58%, Recall@10: 8.27%
weight=0.3 — Hit-rate@10: 42.30%, Precision@10: 6.38%, Recall@10: 8.12%
weight=0.4 — Hit-rate@10: 40.61%, Precision@10: 6.31%, Recall@10: 8.32%
weight=0.5 — Hit-rate@10: 39.59%, Precision@10: 6.01%, Recall@10: 7.97%


One Clear pattern that is visible here is that as the content weight increases , the performace of the Hybrid model decreases.
This confirms the hybrid approach adds real value — but only with a small, carefully
tuned content contribution, not an even split.

| Model | Hit-rate@10 | Precision@10 | Recall@10 |
|---|---|---|---|
| Popularity Baseline | 31.13% | 5.60% | 5.05% |
| Collaborative Filtering | 42.13% | 6.82% | 7.97% |
| Hybrid (weight=0.1) | 43.65% | 7.01% | 8.56% |

**Each stage of the model improves on the last, validating the project's core hypothesis:** personalization (CF) meaningfully beats popularity, and a carefully-weighted hybrid (adding a small content-based signal) beats CF alone.

In [18]:
pure_content_recs = {user: get_hybrid_recommendations(user, k=10, weight=1.0) for user in all_users}
pc_hit = get_hits(pure_content_recs, held_out_liked, k=10)
pc_prec, pc_rec = get_precision_recall(pure_content_recs, held_out_liked, k=10)
print(f"Pure Content-Based — Hit-rate@10: {pc_hit:.2%}, Precision@10: {pc_prec:.2%}, Recall@10: {pc_rec:.2%}")

Pure Content-Based — Hit-rate@10: 5.25%, Precision@10: 0.54%, Recall@10: 0.68%


Content-based similarity built purely from genre tags is a very weak standalone signal for this dataset — even weaker than a naive popularity ranking. It only adds value in small doses (10% weight) as a supplement to collaborative filtering, likely by nudging rankings for movies where genre similarity happens to align with actual taste, without dominating the stronger CF signa

In [16]:
get_hybrid_recommendations(1, k=10, weight=0.1)

[1036, 588, 2294, 858, 1221, 4571, 2081, 780, 3438, 3114]